# Similarity matrices

Builds subject-by-subject correlation matrices from scaffolds and FC (thresholds 100/25/1%) for the 100 unrelated subjects (Schaefer300, Glasser379, AAL89, Shen268, Schaefer100) and plots identifiability (supplementary atlas figures).
The 1200-subject matrices are computed by `scripts/06_similarity_matrices.py`.

In [ ]:
import os
import sys
import sys
sys.path.append("..")
from topofinger import config
from topofinger.hcp_IO import load_scaffold_schaefer300, load_scaffold, save_distances_subjects, save_correlations_subjects, save_correlations_subjects_fast
from topofinger.utils import apply_threshold_to_nested_dict
from topofinger.fingerprinting import compute_idiff, compute_effect_size, compute_success_rate
from topofinger.viz import plot_identifiability

import pickle as pk
from tqdm.notebook import tqdm
import numpy as np

import matplotlib.pyplot as plt
from pingouin import compute_effsize
from scipy.stats import pearsonr

from scipy import stats


%load_ext autoreload
%autoreload 2

os.makedirs(config.FIGURES_RAW + "/supplementary", exist_ok=True)


# Save correlations/distances

## Schaefer300 (100 unrelated)

In [ ]:
data_dir = config.HCP_100 + '/schaefer300/no_hp/'
scaffold_dir = 'scaffolds/whole/'
sc = load_scaffold_schaefer300(data_dir+scaffold_dir+'/freq/')
save_correlations_subjects_fast(sc, data_dir+scaffold_dir+'scaffold_correlations_whole')
fc_dir = 'fc/whole/'
with open(data_dir+'fc/whole/whole_corrs.pck', 'rb') as f:
    fc = pk.load(f)
for th in [100,25,1]:
    fc_th = apply_threshold_to_nested_dict(fc,th)
    #save_distances_subjects(fc_th, data_dir+fc_dir+f'fc_distances_whole_th{th}')
    save_correlations_subjects_fast(fc_th, data_dir+fc_dir+f'fc_correlations_whole_th{th}')

## Glasser (360 cortical regions)

### Save distances and correlations scaffolds

In [ ]:
# Glasser: 360 cortical regions only (19 subcortical regions excluded).
# Same as scripts/06b_similarity_matrices_glasser.py
from topofinger.hcp_IO import load_scaffold_cortical, load_fc_cortical
data_dir = config.HCP_100 + '/glasser379/'
scaffold_dir = 'scaffolds/whole/'
sc = load_scaffold_cortical(data_dir, config.N_CORTICAL_GLASSER, loader=load_scaffold)
save_correlations_subjects_fast(sc, data_dir+scaffold_dir+'scaffold_correlations_whole_cortex')

### Save distances and correlations FC at various threshold

In [ ]:
fc_dir = 'fc/whole/'

fc = load_fc_cortical(data_dir, config.N_CORTICAL_GLASSER)
for th in [100,25,1]:
    fc_th = apply_threshold_to_nested_dict(fc,th)
    save_correlations_subjects_fast(fc_th, data_dir+fc_dir+f'fc_correlations_whole_th{th}_cortex')

In [ ]:
sim_matrix = save_correlations_subjects_fast(fc_th, path=None)

#check and remove NaNs columns
nan_rows = np.isnan(sim_matrix).all(axis=1)
nan_cols = np.isnan(sim_matrix).all(axis=0)
nans = nan_cols | nan_rows
sim_matrix = sim_matrix[~nans][:, ~nans]

plt.imshow(sim_matrix)

## Aal89

### Save distances and correlations scaffolds

In [ ]:
data_dir = config.HCP_100 + '/aal89/'
scaffold_dir = 'scaffolds/whole/'
sc = load_scaffold(data_dir+scaffold_dir+'freq/')
#save_distances_subjects(sc, data_dir+scaffold_dir+'scaffold_distances_whole')
save_correlations_subjects_fast(sc, data_dir+scaffold_dir+'scaffold_correlations_whole')

### Save distances and correlations FC at various threshold

In [ ]:
fc_dir = 'fc/whole/'

with open(data_dir+'fc/whole/whole_corrs.pck', 'rb') as f:
    fc = pk.load(f)
for th in [100,25,1]:
    fc_th = apply_threshold_to_nested_dict(fc,th)
    #save_distances_subjects(fc_th, data_dir+fc_dir+f'fc_distances_whole_th{th}')
    save_correlations_subjects_fast(fc_th, data_dir+fc_dir+f'fc_correlations_whole_th{th}')

## Shen268

### Save distances and correlations scaffolds

In [ ]:
data_dir = config.HCP_100 + '/shen268/'
scaffold_dir = 'scaffolds/whole/'
sc = load_scaffold_schaefer300(data_dir+scaffold_dir+'/freq/')
#save_distances_subjects(sc, data_dir+scaffold_dir+'scaffold_distances_whole')
save_correlations_subjects_fast(sc, data_dir+scaffold_dir+'scaffold_correlations_whole')

### SAve distances and correlations FC at various threshold

In [ ]:
fc_dir = 'fc/whole/'

with open(data_dir+'fc/whole/whole_corrs.pck', 'rb') as f:
    fc = pk.load(f)
for th in [100,25,1]:
    fc_th = apply_threshold_to_nested_dict(fc,th)
    #save_distances_subjects(fc_th, data_dir+fc_dir+f'fc_distances_whole_th{th}')
    save_correlations_subjects_fast(fc_th, data_dir+fc_dir+f'fc_correlations_whole_th{th}')

## Schaefer100

### Save distances and correlations scaffolds

In [ ]:
data_dir = config.HCP_100 + '/schaefer100/'
scaffold_dir = 'scaffolds/whole/'
sc = load_scaffold(data_dir+scaffold_dir+'freq/')
save_distances_subjects(sc, data_dir+scaffold_dir+'scaffold_distances_whole')
save_correlations_subjects_fast(sc, data_dir+scaffold_dir+'scaffold_correlations_whole')

### Save distances and correlations FC at various threshold

In [ ]:
fc_dir = 'fc/whole/'

with open(data_dir+'fc/whole/whole_corrs.pck', 'rb') as f:
    fc = pk.load(f)
for th in [100,25,1]:
    fc_th = apply_threshold_to_nested_dict(fc,th)
    print(np.count_nonzero(fc_th['100307'][0])/fc_th['100307'][0].shape[0]**2)
    save_distances_subjects(fc_th, data_dir+fc_dir+f'fc_distances_whole_th{th}')
    save_correlations_subjects_fast(fc_th, data_dir+fc_dir+f'fc_correlations_whole_th{th}')

# Visualization

## Correlations

In [ ]:
data_dir = config.HCP_100 + '/schaefer300/no_hp/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'


sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')
fc100 = np.load(data_dir+fc_dir+'fc_correlations_whole_th100.npy')
fc25 = np.load(data_dir+fc_dir+'fc_correlations_whole_th25.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1.npy')



fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharey=True, figsize=(20,5))

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.min(mat) for mat in all_matrices])
vmax = max([np.max(mat) for mat in all_matrices])
vmax= 0.65


plot_identifiability(fc100, title = "FC full",ax=ax1, vmin=vmin, vmax=vmax, cmap=cmap);
plot_identifiability(fc25, title = "FC, th=25%", ax=ax2, vmin=vmin, vmax=vmax, cmap=cmap);
plot_identifiability(fc1, title = "FC, th=1%",ax=ax3, vmin=vmin, vmax=vmax, cmap=cmap);
plot_identifiability(sc, title = "Scaffolds", ax=ax4, vmin=vmin, vmax=vmax, cmap=cmap);
fig.suptitle('Identifiability based on Atlas Schaefer 300', fontsize=22);

In [ ]:
data_dir = config.HCP_100 + '/schaefer300/no_hp/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'


sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')
fc100 = np.load(data_dir+fc_dir+'fc_correlations_whole_th100.npy')
fc25 = np.load(data_dir+fc_dir+'fc_correlations_whole_th25.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1.npy')



fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharey=True, figsize=(20,5))

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.min(mat) for mat in all_matrices])
vmax = max([np.max(mat) for mat in all_matrices])
vmax= 0.65

plot_identifiability(fc100, title = "FC full",ax=ax1);
plot_identifiability(fc25, title = "FC, th=25%", ax=ax2);
plot_identifiability(fc1, title = "FC, th=1%",ax=ax3);
plot_identifiability(sc, title = "Scaffolds", ax=ax4);

#plt.savefig(f"{config.FIGURES_RAW}/supplementary/identifiability.png", dpi=250, bbox_inches="tight", facecolor="w")
#fig.suptitle('Identifiability based on Atlas Schaefer 300', fontsize=22);

In [ ]:
sc_inside = np.diag(sc)
sc_outside = np.ravel(sc[~np.eye(sc.shape[0], dtype=bool)])

fc_inside = np.diag(fc1)
fc_outside = np.ravel(fc1[~np.eye(fc1.shape[0], dtype=bool)])

In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))

# First subplot: fc_inside vs fc_outside
ax1.boxplot([fc_inside, fc_outside], tick_labels=['intra-subject', 'inter-subject'], showfliers=False)
ax1.set_title('Functional Connectivity')
ax1.set_ylabel('Correlation')

# Second subplot: sc_inside vs sc_outside
ax2.boxplot([sc_inside, sc_outside], tick_labels=['intra-subject', 'inter-subject'], showfliers=False)
ax2.set_title('Scaffolds')
#ax2.set_ylabel('Correlation')

plt.tight_layout(
)

plt.show()


In [ ]:
data_dir = config.HCP_100 + '/glasser379/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'


sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole_cortex.npy')
fc100 = np.load(data_dir+fc_dir+'fc_correlations_whole_th100_cortex.npy')
fc25 = np.load(data_dir+fc_dir+'fc_correlations_whole_th25_cortex.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1_cortex.npy')



fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharey=False, figsize=(20,5))

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.nanmin(mat) for mat in all_matrices])
vmax = max([np.nanmax(mat) for mat in all_matrices])
vmax= 0.65

plot_identifiability(fc100, title = "FC full",ax=ax1, vmin=vmin, vmax=vmax, cmap=cmap);
plot_identifiability(fc25, title = "FC, th=25%", ax=ax2, vmin=vmin, vmax=vmax, cmap=cmap);
plot_identifiability(fc1, title = "FC, th=1%",ax=ax3, vmin=vmin, vmax=vmax, cmap=cmap);
plot_identifiability(sc, title = "Scaffolds", ax=ax4, vmin=vmin, vmax=vmax, cmap=cmap);
fig.suptitle('Identifiability based on Atlas Glasser 360 (cortical)', fontsize=22);

#save as pdf
plt.savefig(f"{config.FIGURES_RAW}/supplementary/identifiability_glasser.pdf", dpi=250, bbox_inches="tight", facecolor="w")

In [ ]:
data_dir = config.HCP_100 + '/shen268/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'


sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')
fc100 = np.load(data_dir+fc_dir+'fc_correlations_whole_th100.npy')
fc25 = np.load(data_dir+fc_dir+'fc_correlations_whole_th25.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1.npy')



fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharey=True, figsize=(20,5))

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.min(mat) for mat in all_matrices])
vmax = max([np.max(mat) for mat in all_matrices])
vmax= 0.65

plot_identifiability(fc100, title = "FC full",ax=ax1);
plot_identifiability(fc25, title = "FC, th=25%", ax=ax2);
plot_identifiability(fc1, title = "FC, th=1%",ax=ax3);
plot_identifiability(sc, title = "Scaffolds", ax=ax4);
fig.suptitle('Identifiability based on Atlas Shen 268', fontsize=22);

#save as pdf
plt.savefig(f"{config.FIGURES_RAW}/supplementary/identifiability_shen.pdf", dpi=250, bbox_inches="tight", facecolor="w")

In [ ]:
data_dir = config.HCP_100 + '/glasser379/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'

sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole_cortex.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1_cortex.npy')

fig, (ax1, ax2) = plt.subplots(1, 2, sharey=False, figsize=(10,5))
plot_identifiability(fc1, title = "FC, th=1%",ax=ax1);
plot_identifiability(sc, title = "Scaffolds", ax=ax2);
fig.suptitle('Identifiability based on Atlas Glasser 360 (cortical)', fontsize=22);

In [ ]:
data_dir = config.HCP_100 + '/shen268/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'

sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1.npy')

fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(10,5))
plot_identifiability(fc1, title = "FC, th=1%",ax=ax1);
plot_identifiability(sc, title = "Scaffolds", ax=ax2);
fig.suptitle('Identifiability based on Atlas Shen 268', fontsize=22);

In [ ]:
data_dir = config.HCP_100 + '/aal89/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'

sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')
fc100 = np.load(data_dir+fc_dir+'fc_correlations_whole_th100.npy')
fc25 = np.load(data_dir+fc_dir+'fc_correlations_whole_th25.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1.npy')



fig, (ax1, ax2,ax3, ax4) = plt.subplots(1, 4, sharey=True, figsize=(20,5))

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.min(mat) for mat in all_matrices])
vmax = max([np.max(mat) for mat in all_matrices])
vmax= 0.65


plot_identifiability(fc100, title = "FC full",ax=ax1);
plot_identifiability(fc25, title = "FC, th=25%", ax=ax2);
plot_identifiability(fc1, title = "FC, th=1%", ax=ax3);
plot_identifiability(sc, title = "Scaffolds", ax=ax4);
fig.suptitle('Identifiability based on Atlas AAL 90', fontsize=22);

#save as pdf
plt.savefig(f"{config.FIGURES_RAW}/supplementary/identifiability_aal.pdf", dpi=250, bbox_inches="tight", facecolor="w")

In [ ]:
data_dir = config.HCP_100 + '/schaefer100/'
scaffold_dir = 'scaffolds/whole/'
fc_dir = 'fc/whole/'


sc = np.load(data_dir+scaffold_dir+'scaffold_correlations_whole.npy')
fc100 = np.load(data_dir+fc_dir+'fc_correlations_whole_th100.npy')
fc25 = np.load(data_dir+fc_dir+'fc_correlations_whole_th25.npy')
fc1 = np.load(data_dir+fc_dir+'fc_correlations_whole_th1.npy')



fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharey=True, figsize=(20,5))

cmap = 'YlGnBu_r'

# Find global min and max values across all matrices to set the same color scale
all_matrices = [fc100, fc25, fc1, sc]
vmin = min([np.min(mat) for mat in all_matrices])
vmax = max([np.max(mat) for mat in all_matrices])
vmax= 0.65


plot_identifiability(fc100, title = "FC full",ax=ax1);
plot_identifiability(fc25, title = "FC, th=25%", ax=ax2);
plot_identifiability(fc1, title = "FC, th=1%",ax=ax3);
plot_identifiability(sc, title = "Scaffolds", ax=ax4);
fig.suptitle('Identifiability based on Atlas Schaefer 100', fontsize=22);

#save as pdf
plt.savefig(f"{config.FIGURES_RAW}/supplementary/identifiability_schaefer100.pdf", dpi=250, bbox_inches="tight", facecolor="w")